In [1]:
import pandas as pd
from transformers import AutoTokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained("nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16")

In [3]:
data = pd.read_csv("../data/solved/train-cot.csv")

In [4]:
data["token_len"] = data.generated_cot.apply(tokenizer.encode).apply(len)
data["token_len"].describe()

count    7945.000000
mean     1080.042542
std      1390.503863
min       110.000000
25%       224.000000
50%       595.000000
75%       818.000000
max      5475.000000
Name: token_len, dtype: float64

In [5]:
data.groupby("label").token_len.describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
bit manipulation,1602.0,3593.834582,1186.043896,343.0,3312.0,3788.0,4297.0,5475.0
conversion to diff numeral system,1576.0,156.190990,25.320367,110.0,133.0,156.0,174.0,225.0
encryption,1576.0,777.171320,91.785007,531.0,714.0,777.0,840.0,1045.0
gravitational,1597.0,601.629931,47.983251,524.0,545.0,600.0,655.0,670.0
unit conversion,1594.0,245.817440,34.100198,188.0,224.0,237.0,276.0,307.0


In [6]:
data

,id,prompt,answer,prompt_eda,label,generated_cot,computed_answer,is_correct,token_len
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,"To solve this sequence, we will find the speci...",10010111,True,3696
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,"To solve this sequence, we will find the speci...",00011011,False,3917
2,0031df9c,"In Alice's Wonderland, a secret bit manipulati...",00110100,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,"To solve this bit manipulation sequence, we wi...",00110100,True,446
3,004ef7c7,"In Alice's Wonderland, a secret bit manipulati...",11111111,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,"To solve this sequence, we will find the speci...",11110111,False,4296
4,00754598,"In Alice's Wonderland, a secret bit manipulati...",11101111,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,"To solve this sequence, we will find the speci...",11101111,True,3941
...,...,...,...,...,...,...,...,...,...
7940,ff2e376c,"In Alice's Wonderland, the gravitational const...",190.03,"In Alice's Wonderland, the gravitational const...",gravitational,"WARNING: This is Wonderland gravity, NOT Earth...",190.03,True,605
7941,ff85238e,"In Alice's Wonderland, the gravitational const...",57.61,"In Alice's Wonderland, the gravitational const...",gravitational,"WARNING: This is Wonderland gravity, NOT Earth...",57.61,True,606
7942,ff90228b,"In Alice's Wonderland, the gravitational const...",32.39,"In Alice's Wonderland, the gravitational const...",gravitational,"WARNING: This is Wonderland gravity, NOT Earth...",32.39,True,596
7943,ff9540e2,"In Alice's Wonderland, the gravitational const...",107.7,"In Alice's Wonderland, the gravitational const...",gravitational,"WARNING: This is Wonderland gravity, NOT Earth...",107.70,False,605


In [ ]:
target_labels = ['bit manipulation']


correct_col = 'is_correct' if 'is_correct' in data.columns else 'is correct'

filtered_data = data[(data['label'].isin(target_labels)) & (data[correct_col] == False)]

sampled_data = filtered_data.groupby('label').sample(n=5, random_state=42).reset_index(drop=True)

for index, row in sampled_data.iterrows():
    print(f"=== Категория: {row['label']} | ID: {row['id']} ===")
    print("--- Промпт (начало) ---")
    print(str(row['prompt']) + "...\n")
    
    print("--- Сгенерированный CoT ---")
    cot_val = row.get('generated_cot', row.get('generated cot', 'Отсутствует'))
    print(cot_val)
    
    print("\n--- Вычисленный ответ ---")
    ans_val = row.get('computed_answer', row.get('computed answer', 'Отсутствует'))
    print(ans_val)
    
    print("\n" + "="*80 + "\n")

=== Категория: bit manipulation | ID: 13eebe61 ===
--- Промпт (начало) ---
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
00010001 -> 00001000
01010011 -> 0011100...

--- Сгенерированный CoT ---
To solve this sequence, we will find the specific rule for each output bit (0 to 7) by verifying hypotheses against all available examples.
Target input bits for reference (Index 0 to 7): [1, 0, 0, 1, 0, 1, 1, 0]

--- Output Bit 0 ---
Hypothesis: The value is always set to 0.
Verification across examples:
  - Ex 1 (00010001 -> 00001000): Constant value is 0. Matches output bit 0 (0).
  - Ex 2 (01010011 -> 00111001): Constant value is 0. Matches output bit 0 (0).
  - Ex 3 (01101100 -> 01100110): Constant value is 0. Matches output bit 0 (0).
  - Ex 4 (11001011 -> 01111100): Constant val